# Week 3 · Class 3 — Aggregates, GROUP BY & JOINs
**AI Training Program · Phase 1: Python Foundations**

---

## What you'll learn
- Aggregate functions: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`
- `GROUP BY` — summarising data by category
- `HAVING` — filtering groups (vs `WHERE` for rows)
- All JOIN types: `INNER`, `LEFT`, `RIGHT`, `FULL OUTER`
- Combining aggregates with joins

---

These two topics — **aggregation** and **joining** — are the engine of real analytical SQL.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("school.db")
cursor = conn.cursor()

# Build a richer schema for this class
cursor.executescript("""
    DROP TABLE IF EXISTS enrollments;
    DROP TABLE IF EXISTS grades;
    DROP TABLE IF EXISTS students;
    DROP TABLE IF EXISTS courses;

    CREATE TABLE students (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name       TEXT    NOT NULL,
        city       TEXT,
        gpa        REAL
    );

    CREATE TABLE courses (
        course_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        title      TEXT NOT NULL,
        dept       TEXT,
        credits    INTEGER DEFAULT 3
    );

    CREATE TABLE enrollments (
        enroll_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id INTEGER REFERENCES students(student_id),
        course_id  INTEGER REFERENCES courses(course_id),
        score      REAL,
        status     TEXT DEFAULT 'active'
    );

    INSERT INTO students (name, city, gpa) VALUES
        ('Aarav Shah',   'Mumbai',    3.8),
        ('Priya Nair',   'Pune',      3.5),
        ('Rohan Mehta',  'Delhi',     2.9),
        ('Sana Khan',    'Mumbai',    3.7),
        ('Dev Sharma',   'Bangalore', 3.1),
        ('Anita Bose',   'Kolkata',   3.9),
        ('Kiran Rao',    'Hyderabad', NULL),
        ('Meera Pillai', 'Mumbai',    3.6),
        ('Arjun Gupta',  'Delhi',     2.7),
        ('Sneha Iyer',   'Chennai',   3.4);

    INSERT INTO courses (title, dept, credits) VALUES
        ('Python Foundations', 'CS',   4),
        ('Machine Learning',   'AI',   4),
        ('SQL & Databases',    'CS',   3),
        ('Deep Learning',      'AI',   4),
        ('MLOps',              'AI',   3),
        ('Data Visualisation', 'Math', 3);

    INSERT INTO enrollments (student_id, course_id, score, status) VALUES
        (1, 1, 88,  'active'),  (1, 2, 75,  'active'),  (1, 3, 91,  'active'),
        (2, 1, 72,  'active'),  (2, 4, 84,  'active'),
        (3, 2, 61,  'active'),  (3, 3, 55,  'dropped'),
        (4, 1, 95,  'active'),  (4, 2, 88,  'active'),  (4, 5, 79,  'active'),
        (5, 3, 70,  'active'),  (5, 4, 66,  'active'),
        (6, 1, 98,  'active'),  (6, 2, 92,  'active'),  (6, 4, 89,  'active'),
        (7, 5, 77,  'active'),
        (8, 1, 83,  'active'),  (8, 3, 76,  'active'),
        (9, 2, 58,  'dropped'), (9, 6, 64,  'active'),
        (10,1, 81,  'active'),  (10,6, 73,  'active');
""")
conn.commit()
print("Database ready ✓")

## Part A: Aggregate Functions

Aggregate functions collapse **many rows** into **one value**.

| Function | Returns |
|---|---|
| `COUNT(*)` | Total number of rows |
| `COUNT(col)` | Rows where col is not NULL |
| `COUNT(DISTINCT col)` | Number of unique non-NULL values |
| `SUM(col)` | Total of all values |
| `AVG(col)` | Mean (ignores NULL) |
| `MIN(col)` | Smallest value |
| `MAX(col)` | Largest value |

In [ ]:
# Overall statistics across all students
pd.read_sql_query("""
    SELECT
        COUNT(*)             AS total_students,
        COUNT(gpa)           AS students_with_gpa,
        ROUND(AVG(gpa), 2)   AS avg_gpa,
        MIN(gpa)             AS min_gpa,
        MAX(gpa)             AS max_gpa,
        COUNT(DISTINCT city) AS unique_cities
    FROM students
""", conn)

In [ ]:
# Enrollment stats
pd.read_sql_query("""
    SELECT
        COUNT(*)            AS total_enrollments,
        SUM(score)          AS total_score_points,
        ROUND(AVG(score),2) AS avg_score,
        MIN(score)          AS lowest_score,
        MAX(score)          AS highest_score
    FROM enrollments
    WHERE status = 'active'
""", conn)

## Part B: `GROUP BY` — Aggregation by Category

`GROUP BY` splits rows into groups and applies the aggregate **per group**.

```
Raw rows         Grouped by city       Result
──────────────   ──────────────────   ──────────────────────
Mumbai  3.8      Mumbai →  3.8         Mumbai  avg=3.7
Mumbai  3.7                3.7
Mumbai  3.6                3.6
Pune    3.5      Pune   →  3.5         Pune    avg=3.5
Delhi   2.9      Delhi  →  2.9         Delhi   avg=2.8
Delhi   2.7                2.7
```

In [ ]:
# Student count and average GPA per city
pd.read_sql_query("""
    SELECT
        city,
        COUNT(*)           AS num_students,
        ROUND(AVG(gpa),2)  AS avg_gpa
    FROM   students
    GROUP BY city
    ORDER BY num_students DESC
""", conn)

In [ ]:
"""
1 - active
2 - active
2 - dropped
3 - active
3 - dropped
4 - active
5 - active
6 - active
"""

In [ ]:
# Enrollments per course: count, avg score
pd.read_sql_query("""
    SELECT
        c.title,
        c.dept,
        COUNT(e.enroll_id)     AS enrolled,
        ROUND(AVG(e.score), 1) AS avg_score
    FROM   courses    c
    LEFT JOIN enrollments e ON c.course_id = e.course_id
    GROUP BY c.course_id, c.title, c.dept
    ORDER BY avg_score DESC
""", conn)

## Part C: `HAVING` — Filtering Groups

| | `WHERE` | `HAVING` |
|---|---|---|
| Filters | **Rows** (before grouping) | **Groups** (after grouping) |
| Can use aggregates? | ❌ No | ✅ Yes |
| Runs | Before `GROUP BY` | After `GROUP BY` |

In [ ]:
# Cities with more than 1 student
pd.read_sql_query("""
    SELECT city, COUNT(*) AS num_students
    FROM   students
    GROUP BY city
    HAVING COUNT(*) > 1
    ORDER BY num_students DESC
""", conn)

In [ ]:
# WHERE vs HAVING together:
# Active enrollments only (WHERE), then courses with avg score > 75 (HAVING)
pd.read_sql_query("""
    SELECT
        c.title,
        COUNT(e.enroll_id)     AS active_enrollments,
        ROUND(AVG(e.score), 1) AS avg_score
    FROM   courses c
    JOIN   enrollments e ON c.course_id = e.course_id
    WHERE  e.status = 'active'          -- filter rows first
    GROUP BY c.course_id, c.title
    HAVING AVG(e.score) > 75            -- then filter groups
    ORDER BY avg_score DESC
""", conn)

---

## Part D: JOINs — Combining Tables

A **JOIN** links rows from two tables based on a matching condition.

```
 students                enrollments
 ─────────               ───────────────────
 student_id  name        enroll_id  student_id  course_id  score
 1           Aarav       1          1           1          88
 2           Priya       2          1           2          75
 3           Rohan       3          2           1          72
                         ↑
            JOIN ON students.student_id = enrollments.student_id
```

### Four main JOIN types

| Type | Returns |
|---|---|
| `INNER JOIN` | Only rows that match in BOTH tables |
| `LEFT JOIN` | All left rows + matching right (NULL if no match) |
| `RIGHT JOIN` | All right rows + matching left (NULL if no match) |
| `FULL OUTER JOIN` | All rows from both, NULL where no match |

### D.1 — `INNER JOIN`
Returns only rows with a match in both tables.

In [ ]:
# Each enrollment with student name and course title
pd.read_sql_query("""
    SELECT
        s.name       AS student,
        c.title      AS course,
        e.score,
        e.status
    FROM   enrollments e
    INNER JOIN students s ON e.student_id = s.student_id
    INNER JOIN courses  c ON e.course_id  = c.course_id
    ORDER BY s.name, c.title
""", conn)

### D.2 — `LEFT JOIN`
Keeps **all rows from the left table**, even if there is no match on the right.

In [ ]:
# All courses, even those with no enrollments
pd.read_sql_query("""
    SELECT
        c.title,
        COUNT(e.enroll_id) AS enrollments
    FROM   courses c
    LEFT JOIN enrollments e ON c.course_id = e.course_id
    GROUP BY c.course_id, c.title
    ORDER BY enrollments DESC
""", conn)

In [ ]:
# Find students NOT enrolled in any course
# Trick: LEFT JOIN + WHERE right side IS NULL
pd.read_sql_query("""
    SELECT s.name, s.city
    FROM   students s
    LEFT JOIN enrollments e ON s.student_id = e.student_id
    WHERE  e.enroll_id IS NULL
""", conn)

### D.3 — `FULL OUTER JOIN` (simulated in SQLite)
SQLite doesn't support FULL OUTER JOIN natively, but we can simulate it.

In [ ]:
# FULL OUTER JOIN = LEFT JOIN UNION ALL RIGHT JOIN (anti-join)
pd.read_sql_query("""
    SELECT s.name AS student, c.title AS course
    FROM   students s
    LEFT JOIN courses c ON s.course_id = c.course_id

    UNION ALL

    SELECT s.name, c.title
    FROM   courses c
    LEFT JOIN students s ON c.course_id = s.course_id
    WHERE  s.course_id IS NULL;
""", conn)

## Part E: Real Analysis — Joining + Aggregating Together

In [ ]:
# Student leaderboard: total score across all active courses
pd.read_sql_query("""
    SELECT
        s.name,
        s.city,
        COUNT(e.course_id)     AS courses_taken,
        ROUND(SUM(e.score), 0) AS total_score,
        ROUND(AVG(e.score), 1) AS avg_score
    FROM   students s
    JOIN   enrollments e ON s.student_id = e.student_id
    WHERE  e.status = 'active'
    GROUP BY s.student_id, s.name, s.city
    ORDER BY avg_score DESC
""", conn)

In [ ]:
# Department summary: courses, enrolled students, avg score
pd.read_sql_query("""
    SELECT
        c.dept,
        COUNT(DISTINCT c.course_id)    AS num_courses,
        COUNT(DISTINCT e.student_id)   AS unique_students,
        ROUND(AVG(e.score), 1)         AS avg_score
    FROM   courses c
    LEFT JOIN enrollments e ON c.course_id = e.course_id
    GROUP BY c.dept
    ORDER BY avg_score DESC
""", conn)

---

## ✏️ Exercises

**Exercise 1**: Count how many students are enrolled in each course (include courses with zero students). Sort by count descending.

In [ ]:
# Your code here


**Exercise 2**: Find cities with an average student GPA above 3.5 (only count students who have a GPA). Show city and avg_gpa.

In [ ]:
# Your code here


**Exercise 3**: List each student's name, number of courses enrolled (active only), and their highest score.

In [ ]:
# Your code here


**Exercise 4 (Challenge)**: Which course has the most students with a score above 80? Show course title and count.

In [ ]:
# Your code here


---

## Summary

```
SELECT   [columns + aggregates]
FROM     table1
JOIN     table2  ON  condition       ← combine tables
WHERE    row_filter                  ← filter before grouping
GROUP BY grouping_columns            ← form groups
HAVING   group_filter                ← filter after grouping
ORDER BY sort_column
LIMIT    n;
```

**Next class**: Subqueries, CTEs (`WITH`), window functions (intro), and Git basics.

In [ ]:
conn.close()